In [1]:
# Test ConsensusLeidenClustering
import igraph as ig
import numpy as np
import pandas as pd
from skclust.graph import ConsensusLeidenClustering, compute_membership_cooccurrence

print("=" * 80)
print("TEST 1: Basic functionality with Zachary's Karate Club")
print("=" * 80)

# Create test graph
graph = ig.Graph.Famous('Zachary')
graph.vs['name'] = [f'node_{i}' for i in range(graph.vcount())]

print(f"\nOriginal graph: {graph.vcount()} nodes, {graph.ecount()} edges")

# Test default settings (RBConfigurationVertexPartition, resolution=1.0)
leiden = ConsensusLeidenClustering(n_iter=10, n_jobs=1, random_state=42, verbose=True)
leiden.fit(graph)

print(f"\nPartitions shape: {leiden.partitions_.shape}")
print(f"Membership matrix shape: {leiden.membership_matrix_.shape}")
print(f"Consensus edges: {len(leiden.consensus_edges_)} / {graph.ecount()} edges")
print(f"Mean consensus ratio: {leiden.consensus_ratio_.mean():.3f}")
print(f"Median consensus ratio: {leiden.consensus_ratio_.median():.3f}")

# Transform to consensus graph
consensus_graph = leiden.transform(graph)
print(f"\nConsensus graph: {consensus_graph.vcount()} nodes, {consensus_graph.ecount()} edges")

print("\n" + "=" * 80)
print("TEST 2: Resolution parameter sweep")
print("=" * 80)

for res in [0.5, 1.0, 1.5, 2.0]:
    leiden_res = ConsensusLeidenClustering(
        n_iter=10,
        resolution_parameter=res,
        n_jobs=-1,
        random_state=42,
        verbose=False
    )
    leiden_res.fit(graph)
    n_clusters_per_iter = leiden_res.partitions_.nunique(axis=0)
    print(f"Resolution {res}: "
          f"consensus edges = {len(leiden_res.consensus_edges_):3d}, "
          f"avg clusters = {n_clusters_per_iter.mean():.1f}")

print("\n" + "=" * 80)
print("TEST 3: Weighted graph")
print("=" * 80)

# Add random weights
np.random.seed(42)
graph.es['weight'] = np.random.uniform(0.1, 1.0, graph.ecount())

leiden_weighted = ConsensusLeidenClustering(
    n_iter=10,
    weight='weight',
    n_jobs=-1,
    random_state=42,
    verbose=False
)
leiden_weighted.fit(graph)

print(f"Weighted consensus edges: {len(leiden_weighted.consensus_edges_)} / {graph.ecount()}")
print(f"Weighted mean consensus: {leiden_weighted.consensus_ratio_.mean():.3f}")

print("\n" + "=" * 80)
print("TEST 4: Different partition types")
print("=" * 80)

from leidenalg import ModularityVertexPartition, CPMVertexPartition

# Modularity
leiden_mod = ConsensusLeidenClustering(
    n_iter=10,
    partition_type=ModularityVertexPartition,
    n_jobs=-1,
    random_state=42,
    verbose=False
)
leiden_mod.fit(graph)
print(f"ModularityVertexPartition: {len(leiden_mod.consensus_edges_)} consensus edges")

# CPM with resolution
leiden_cpm = ConsensusLeidenClustering(
    n_iter=10,
    partition_type=CPMVertexPartition,
    leiden_kws={'resolution_parameter': 0.1},
    n_jobs=-1,
    random_state=42,
    verbose=False
)
leiden_cpm.fit(graph)
print(f"CPMVertexPartition: {len(leiden_cpm.consensus_edges_)} consensus edges")

print("\n" + "=" * 80)
print("TEST 5: Standalone compute_membership_cooccurrence function")
print("=" * 80)

# Use partitions from previous fit
df_partitions = leiden.partitions_
print(f"Input partitions shape: {df_partitions.shape}")

# Compute co-occurrence manually
cooccur = compute_membership_cooccurrence(df_partitions)
print(f"Co-occurrence matrix shape: {cooccur.shape}")
print(f"Co-occurrence data type: {cooccur.dtypes[0]}")

# Verify it matches the class method
assert cooccur.equals(leiden.membership_matrix_), "Mismatch between standalone and class method!"
print("✓ Standalone function matches class method")

# Get perfect consensus pairs
perfect_pairs = cooccur.mean(axis=1)[lambda x: x == 1.0].index
print(f"Perfect consensus pairs: {len(perfect_pairs)}")

print("\n" + "=" * 80)
print("TEST 6: Parallel vs Sequential consistency")
print("=" * 80)

# Sequential
leiden_seq = ConsensusLeidenClustering(
    n_iter=10,
    n_jobs=1,
    random_state=999,
    verbose=False
)
leiden_seq.fit(graph)

# Parallel
leiden_par = ConsensusLeidenClustering(
    n_iter=10,
    n_jobs=-1,
    random_state=999,
    verbose=False
)
leiden_par.fit(graph)

# Compare results
assert leiden_seq.partitions_.equals(leiden_par.partitions_), "Sequential != Parallel partitions!"
assert leiden_seq.consensus_edges_ == leiden_par.consensus_edges_, "Sequential != Parallel consensus!"
print("✓ Sequential and parallel execution produce identical results")

print("\n" + "=" * 80)
print("TEST 7: fit_transform shortcut")
print("=" * 80)

leiden_ft = ConsensusLeidenClustering(n_iter=10, n_jobs=-1, random_state=42, verbose=False)
consensus_graph_ft = leiden_ft.fit_transform(graph)

print(f"fit_transform consensus graph: {consensus_graph_ft.vcount()} nodes, {consensus_graph_ft.ecount()} edges")
print("✓ fit_transform works")

print("\n" + "=" * 80)
print("TEST 8: Edge cases and error handling")
print("=" * 80)

# Test transform before fit
leiden_error = ConsensusLeidenClustering()
try:
    leiden_error.transform(graph)
    print("✗ Should have raised RuntimeError")
except RuntimeError as e:
    print(f"✓ Correct error on transform before fit: {e}")

# Test graph without 'name' attribute
graph_no_name = ig.Graph.Famous('Zachary')
try:
    leiden_error.fit(graph_no_name)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Correct error on missing 'name': {e}")

# Test invalid weight attribute
leiden_bad_weight = ConsensusLeidenClustering(weight='nonexistent')
try:
    leiden_bad_weight.fit(graph)
    print("✗ Should have raised ValueError")
except ValueError as e:
    print(f"✓ Correct error on invalid weight: {e}")

print("\n" + "=" * 80)
print("TEST 9: Consensus ratio distribution")
print("=" * 80)

leiden_dist = ConsensusLeidenClustering(n_iter=50, n_jobs=-1, random_state=42, verbose=False)
leiden_dist.fit(graph)

print("\nConsensus ratio distribution:")
print(leiden_dist.consensus_ratio_.describe())

# Show some example edges with different consensus levels
print("\nExample edges by consensus level:")
for threshold in [1.0, 0.9, 0.8, 0.5]:
    edges_at_threshold = leiden_dist.consensus_ratio_[
        (leiden_dist.consensus_ratio_ >= threshold) & 
        (leiden_dist.consensus_ratio_ < threshold + 0.1)
    ]
    if len(edges_at_threshold) > 0:
        print(f"  {threshold:.1f}-{threshold+0.1:.1f}: {len(edges_at_threshold)} edges")

print("\n" + "=" * 80)
print("ALL TESTS PASSED! ✓")
print("=" * 80)

TEST 1: Basic functionality with Zachary's Karate Club

Original graph: 34 nodes, 78 edges


Leiden clustering:   0%|          | 0/10 [00:00<?, ?it/s]


Partitions shape: (34, 10)
Membership matrix shape: (561, 10)
Consensus edges: 146 / 78 edges
Mean consensus ratio: 0.260
Median consensus ratio: 0.000

Consensus graph: 34 nodes, 57 edges

TEST 2: Resolution parameter sweep
Resolution 0.5: consensus edges = 272, avg clusters = 2.0
Resolution 1.0: consensus edges = 146, avg clusters = 4.0
Resolution 1.5: consensus edges =  81, avg clusters = 5.0
Resolution 2.0: consensus edges =  45, avg clusters = 7.3

TEST 3: Weighted graph
Weighted consensus edges: 156 / 78
Weighted mean consensus: 0.278

TEST 4: Different partition types
ModularityVertexPartition: 146 consensus edges
CPMVertexPartition: 139 consensus edges

TEST 5: Standalone compute_membership_cooccurrence function
Input partitions shape: (34, 10)
Co-occurrence matrix shape: (561, 10)
Co-occurrence data type: bool
✓ Standalone function matches class method
Perfect consensus pairs: 146

TEST 6: Parallel vs Sequential consistency
✓ Sequential and parallel execution produce identica